# Gold layer EDA — 50K reviews (feature + label stores)

**Owner:** Van (Modeler)  
**Data:** Charlie's local gold export under `data/local/gold/` (gitignored)

This notebook loads partitioned Parquet from:
- `feature_store_50K/` — `review_id`, `review_date`, `text`
- `label_store_50K/` — `review_id`, `review_date`, `label`

## Pandas vs PySpark?

**Use Pandas here.** ~50K rows fits easily in memory (a few MB of text). Pandas + PyArrow reads Parquet natively, runs in a local notebook with zero cluster setup, and matches what `models/train.py` expects downstream.

Reach for PySpark when data is **millions+ rows**, lives on a distributed lake, or you need cluster-scale transforms. For this dataset it adds JVM overhead without benefit.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Resolve repo root whether the kernel cwd is repo root or notebooks/
ROOT = Path.cwd()
if not (ROOT / "data" / "local").exists() and (ROOT.parent / "data" / "local").exists():
    ROOT = ROOT.parent

GOLD_DIR = ROOT / "data" / "local" / "gold"
FEATURE_DIR = GOLD_DIR / "feature_store_50K"
LABEL_DIR = GOLD_DIR / "label_store_50K"

CANONICAL_LABELS = ["negative", "neutral", "positive"]

print(f"Repo root: {ROOT}")
print(f"Feature dir exists: {FEATURE_DIR.exists()}")
print(f"Label dir exists: {LABEL_DIR.exists()}")

## 1. Loading Parquet

Charlie's export is **Hive-style partitioned**: one folder per `review_date`, each with a `part.parquet` file.

Three equivalent patterns:
1. **Glob all files** and `pd.concat` — explicit, works everywhere.
2. **`pd.read_parquet(directory)`** — pandas 2.x reads a partitioned dataset when `pyarrow` is installed.
3. **Single partition** — useful for spot-checking one day before loading everything.

We use option 1 below so the steps are visible.

In [ ]:
def load_parquet_store(root: Path) -> tuple[pd.DataFrame, int]:
    """Read all *.parquet files under a partitioned store directory."""
    files = sorted(root.rglob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parquet files under {root}")
    frames = [pd.read_parquet(path) for path in files]
    return pd.concat(frames, ignore_index=True), len(files)


# Spot-check: one partition before loading everything
sample_path = next(FEATURE_DIR.rglob("*.parquet"))
sample_df = pd.read_parquet(sample_path)
print(f"Sample file: {sample_path.relative_to(ROOT)}")
display(sample_df.head(3))
print(f"Columns: {list(sample_df.columns)}")

In [ ]:
features, n_feature_files = load_parquet_store(FEATURE_DIR)
labels, n_label_files = load_parquet_store(LABEL_DIR)

print(f"Feature partitions read: {n_feature_files:,}")
print(f"Label partitions read:   {n_label_files:,}")
print(f"Feature rows: {len(features):,}")
print(f"Label rows:   {len(labels):,}")

display(features.head(3))
display(labels.head(3))

## 2. Join feature + label stores

Join on `(review_id, review_date)` — the natural key for this export. After join we have the training-ready `(text, label)` pairs.

In [ ]:
JOIN_KEYS = ["review_id", "review_date"]

df = features.merge(labels, on=JOIN_KEYS, how="inner", validate="one_to_one")

left_only = features.merge(labels, on=JOIN_KEYS, how="left", indicator=True)
right_only = labels.merge(features, on=JOIN_KEYS, how="left", indicator=True)

print(f"Joined rows: {len(df):,}")
print(f"Feature-only rows (left anti): {(left_only['_merge'] == 'left_only').sum():,}")
print(f"Label-only rows (right anti):  {(right_only['_merge'] == 'left_only').sum():,}")

display(df.head(5))

## 3. Schema & dtypes

In [ ]:
df.info()
display(df.describe(include="all").T)

## 4. Label distribution

Compare against `models.baseline_sklearn.LABELS` (`negative`, `neutral`, `positive`). Class imbalance affects baseline tuning and metric choice (macro F1 vs accuracy).

In [ ]:
label_counts = df["label"].value_counts().reindex(CANONICAL_LABELS)
label_pct = (label_counts / len(df) * 100).round(1)
label_summary = pd.DataFrame({"count": label_counts, "pct": label_pct})
display(label_summary)

unexpected = sorted(set(df["label"].unique()) - set(CANONICAL_LABELS))
print(f"Unexpected label values: {unexpected or 'none'}")

fig, ax = plt.subplots(figsize=(6, 4))
label_counts.plot(kind="bar", ax=ax, color=["#d62728", "#ffbf00", "#2ca02c"])
ax.set_title("Label distribution (50K gold export)")
ax.set_xlabel("label")
ax.set_ylabel("count")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 5. Text length

Useful for TF-IDF `max_features`, DistilBERT `max_length`, and spotting truncation risk.

In [ ]:
df["text_len"] = df["text"].str.len()

display(df["text_len"].describe().to_frame().T)
print(f"Reviews at max length (5000 chars): {(df['text_len'] == 5000).sum():,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["text_len"].hist(bins=50, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Text length (characters)")
axes[0].set_xlabel("chars")
axes[0].set_ylabel("count")

for lbl, color in zip(CANONICAL_LABELS, ["#d62728", "#ffbf00", "#2ca02c"]):
    subset = df.loc[df["label"] == lbl, "text_len"]
    axes[1].hist(subset, bins=40, alpha=0.5, label=lbl, color=color)
axes[1].set_title("Text length by label")
axes[1].set_xlabel("chars")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Reviews over time

Partition folders are keyed by `review_date` — check volume and label mix over the timeline.

In [ ]:
df["review_date"] = pd.to_datetime(df["review_date"])

daily = (
    df.groupby(["review_date", "label"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CANONICAL_LABELS, fill_value=0)
)

fig, ax = plt.subplots(figsize=(12, 4))
daily.plot(kind="area", stacked=True, ax=ax, color=["#d62728", "#ffbf00", "#2ca02c"], alpha=0.8)
ax.set_title("Daily review volume by label")
ax.set_xlabel("review_date")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

display(daily.sum(axis=1).describe().to_frame("reviews_per_day").T)

## 7. Sample reviews per label

Eyeball a few rows to sanity-check label quality before training.

In [ ]:
for lbl in CANONICAL_LABELS:
    print(f"\n{'=' * 60}\n{lbl.upper()} — random sample\n{'=' * 60}")
    sample = df.loc[df["label"] == lbl].sample(2, random_state=42)
    for _, row in sample.iterrows():
        preview = row["text"][:300].replace("\n", " ")
        print(f"[{row['review_date'].date()}] ({row['text_len']} chars)")
        print(preview + ("…" if len(row["text"]) > 300 else ""))
        print()

## 8. Data quality checklist

Quick gates before passing this to `models/train.py` or `models/distilbert_finetune.py`.

In [ ]:
checks = {
    "total_rows": len(df),
    "null_text": int(df["text"].isna().sum()),
    "null_label": int(df["label"].isna().sum()),
    "empty_text": int((df["text"].str.strip() == "").sum()),
    "duplicate_review_id": int(df["review_id"].duplicated().sum()),
    "labels_in_canonical_set": sorted(set(df["label"].unique()) - set(CANONICAL_LABELS)) == [],
    "date_range": f"{df['review_date'].min().date()} → {df['review_date'].max().date()}",
}

for k, v in checks.items():
    print(f"{k:28} {v}")

# Training-ready export (optional — also gitignored under data/local/)
# review_date is kept so the tuning notebooks can build temporal OOT/demo splits.
TRAINING_CSV = GOLD_DIR / "gold_50k_training.csv"
df[["text", "label", "review_date"]].to_csv(TRAINING_CSV, index=False)
print(f"\nWrote training CSV: {TRAINING_CSV.relative_to(ROOT)} ({len(df):,} rows)")
print("Run: python models/train.py --data data/local/gold/gold_50k_training.csv")